# Machine Learning Workflow

Machine learning is not a single algorithm — it is a discipline with a repeatable process for turning data into working systems. Knowing individual algorithms is not enough. Production-quality models require a disciplined workflow: defining the problem clearly, exploring and cleaning data, training and tuning models fairly, evaluating rigorously, and deploying with monitoring.

This notebook covers the **end-to-end machine learning workflow** — the capstone of classical machine learning. It ties together preprocessing, modeling, evaluation, and deployment into a process you can apply to any project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import os
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, confusion_matrix)

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Overview of the ML Workflow

A successful machine learning project follows a structured process. Skipping steps — especially problem definition, data quality checks, or proper evaluation — leads to models that fail in production.

```
  1. Define Problem → 2. Collect Data → 3. Explore Data
         ↑                                        ↓
  8. Monitor ← 7. Deploy ← 6. Evaluate ← 4. Preprocess
         ↑                              ↓
         └──────── 5. Train & Tune ─────┘
```

This is not strictly linear — you iterate. Exploration reveals preprocessing needs; evaluation reveals modeling gaps; monitoring reveals drift that sends you back to retraining.

---

## 2. Step 1: Define the Problem

Before touching data, answer:

- **What decision will the model support?** Predict customer churn, classify images, recommend products.
- **What does success look like?** 95% recall on fraud detection, RMSE below 5% on price prediction.
- **Is machine learning the right approach?** Can rules solve it? Is labeled data available?
- **What are the constraints?** Latency requirements, interpretability needs, fairness considerations, budget.
- **What metric matches the business goal?** Accuracy is wrong for imbalanced data; precision matters when false alarms are costly.

A well-defined problem prevents months of work on a model that solves the wrong question.

---

## 3. Step 2: Collect and Explore Data

**Collect** data from databases, APIs, files, sensors, or public datasets. Document sources, collection dates, and known limitations.

**Explore** with summary statistics and visualizations:

- How many samples? How many features?
- Missing values? Outliers? Duplicates?
- Feature distributions? Class balance?
- Correlations between features?
- Does the data match what you expect from domain knowledge?

Exploration often reveals data quality issues that must be fixed before modeling. It also informs feature engineering ideas and helps you understand whether the problem is feasible with available data.

In [ ]:
# Exploratory data analysis on breast cancer dataset
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df["target"] = cancer.target
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})

print(f"Shape: {df.shape}")
print(f"\nClass distribution:\n{df['diagnosis'].value_counts()}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nFeature statistics (first 5):")
print(df[cancer.feature_names[:5]].describe().round(2))

---

## 4. Step 3: Preprocess and Engineer Features

Transform raw data into a format models can use:

- Handle missing values (impute or remove).
- Encode categorical variables (one-hot, label encoding).
- Scale numerical features (standardize or normalize).
- Engineer new features from domain knowledge.
- Remove duplicates and fix errors.

Build preprocessing into a **Pipeline** so the same transformations are applied consistently during training and inference. Fit preprocessing on training data only to prevent leakage.

---

## 5. Step 4: Train and Tune Models

1. **Establish a baseline** — a simple model (majority class, mean prediction, logistic regression) sets the performance floor.
2. **Try multiple algorithms** — compare with cross-validation on the training set.
3. **Tune hyperparameters** — grid search or random search with cross-validation.
4. **Select the best model** based on validation performance and practical constraints (speed, interpretability).

Never select or tune models using the test set. The test set is reserved for one final evaluation.

In [ ]:
# Model training and hyperparameter tuning
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=42))
])

param_grid = {
    "clf__n_estimators": [50, 100, 200],
    "clf__max_depth": [3, 5, 8, None],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV ROC-AUC: {grid.best_score_:.4f}")

---

## 6. Step 5: Evaluate

Evaluate the final tuned model on the **held-out test set** — once, at the end.

Report metrics aligned with the business problem. Analyze errors: which samples are misclassified? Are errors concentrated in a subgroup? Is the model calibrated (predicted probabilities match actual frequencies)?

Document results, limitations, and assumptions. A model that works on test data from 2023 may fail on 2026 data if the world changes.

In [ ]:
# Final evaluation on test set
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Test Set Evaluation")
print("=" * 40)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=cancer.target_names)}")

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(cancer.target_names); ax.set_yticklabels(cancer.target_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
ax.set_title("Confusion Matrix (Test Set)")
plt.show()

---

## 7. Step 6: Deploy and Monitor

**Deployment** makes the model available for predictions:

- **Batch prediction** — score large datasets offline (nightly fraud scoring, monthly churn prediction).
- **Real-time API** — serve predictions on demand (recommendation at click time, image classification).
- **Edge deployment** — run on device (mobile, IoT sensor).

Save the entire pipeline (preprocessing + model) as a single artifact so inference applies the same transformations as training.

**Monitoring** in production:

- **Data drift** — input feature distributions shifting from training data.
- **Concept drift** — the relationship between features and target changing over time.
- **Performance degradation** — accuracy dropping on recent data.
- **Latency and throughput** — meeting service-level requirements.

When drift or degradation is detected, retrain the model on fresh data and redeploy.

In [ ]:
# Save and load pipeline for deployment
model_path = "ml_pipeline_model.joblib"
joblib.dump(best_model, model_path)

loaded_model = joblib.load(model_path)
sample = X_test[:3]
predictions = loaded_model.predict(sample)
probabilities = loaded_model.predict_proba(sample)

print("Saved and loaded pipeline successfully")
print(f"\nSample predictions: {predictions}")
print(f"Sample probabilities:\n{np.round(probabilities, 3)}")

os.remove(model_path)
print(f"\nCleaned up {model_path}")

---

## 8. End-to-End Workflow Example

The following function encapsulates the complete workflow — from raw data to evaluated model — in a reproducible sequence.

In [ ]:
def ml_workflow(X, y, model, param_grid=None, test_size=0.2, scoring="accuracy"):
    """End-to-end ML workflow: split → tune → evaluate."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42)

    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])

    if param_grid:
        search = GridSearchCV(pipe, param_grid, cv=5, scoring=scoring)
        search.fit(X_train, y_train)
        final_model = search.best_estimator_
        print(f"Best params: {search.best_params_}")
        print(f"CV {scoring}: {search.best_score_:.4f}")
    else:
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring=scoring)
        pipe.fit(X_train, y_train)
        final_model = pipe
        print(f"CV {scoring}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    y_pred = final_model.predict(X_test)
    test_score = accuracy_score(y_test, y_pred)
    print(f"Test accuracy: {test_score:.4f}")
    return final_model, test_score

model, score = ml_workflow(
    cancer.data, cancer.target,
    RandomForestClassifier(random_state=42),
    param_grid={"model__n_estimators": [50, 100, 200],
                "model__max_depth": [3, 5, 8]},
    scoring="roc_auc"
)

---

## 9. Common Pitfalls and Best Practices

**Pitfalls to avoid:**

- Training on the test set (data leakage).
- Optimizing accuracy on imbalanced data.
- Skipping baseline models.
- Over-engineering before validating the problem is solvable.
- Ignoring data quality issues.
- Deploying without monitoring.
- Assuming the model will work forever without retraining.

**Best practices:**

- Start simple, add complexity only when justified by validation metrics.
- Use pipelines for reproducibility.
- Document data sources, preprocessing, and model versions.
- Version control code, data snapshots, and model artifacts.
- Align metrics with business objectives.
- Plan for retraining before deployment, not after failure.
- Test edge cases and adversarial inputs.

---

## 10. The ML Project Checklist

Use this checklist for any machine learning project:

- [ ] Problem clearly defined with success metrics
- [ ] Data collected and documented
- [ ] Exploratory analysis completed
- [ ] Data quality issues addressed
- [ ] Train/validation/test split created (stratified if needed)
- [ ] Preprocessing pipeline built and fitted on train only
- [ ] Baseline model established
- [ ] Multiple models compared with cross-validation
- [ ] Hyperparameters tuned on validation set
- [ ] Final model evaluated once on test set
- [ ] Error analysis performed
- [ ] Model and pipeline saved for deployment
- [ ] Monitoring plan defined
- [ ] Retraining schedule established

---

## 11. Summary

The **machine learning workflow** provides a repeatable path from problem to production: define the problem, explore data, preprocess, train and tune models, evaluate rigorously, deploy, and monitor. Each step builds on the core techniques of the field — supervised and unsupervised learning, preprocessing, evaluation, ensembles, and regularization.

Machine learning is an iterative discipline. Models improve through cycles of experimentation, evaluation, and refinement. A disciplined workflow — not just a powerful algorithm — is what separates models that work in notebooks from systems that deliver value in the real world.